In [ ]:
!pip install torch numpy pandas matplotlib seaborn tqdm transformers datasets peft bitsandbytes accelerate scikit-learn

In [ ]:
!pip install ipywidgets widgetsnbextension

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [ ]:
"""
Обучение LLM с помощью QLoRA для классификации запросов.
Исправленная версия: стабильный 8-bit LoRA + корректный Trainer + O(N) evaluation.
Добавлены детальный промпт и графики (сравнение accuracy, confusion matrices).
"""

import os
#os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import gc
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    BitsAndBytesConfig,
    EarlyStoppingCallback,
    AutoModelForSequenceClassification
)

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

# ================== НАСТРОЙКИ ==================
TRAIN_PATH = "train.csv"
TEST_PATH = "test.csv"

MAX_LENGTH = 512
OUTPUT_DIR = "./llm_classifier"
PLOTS_DIR = "./llm_plots"
RESULTS_CSV = "results.csv"

os.makedirs(PLOTS_DIR, exist_ok=True)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

MODEL_NAMES = [
    "mistralai/Mistral-7B-Instruct-v0.3",
    "apple/SimpleSD-4B-instruct",
    "Qwen/Qwen3-4B-Instruct-2507",
    "Qwen/Qwen2.5-7B-Instruct",
    "unsloth/Meta-Llama-3.1-8B-Instruct",
    "openchat/openchat-3.6-8b-20240522",
    "google/gemma-4-E2B-it",
    "google/gemma-4-E4B-it",
    "AvitoTech/avibe",
]

DIGIT2CLASS = {
    "1": "теоретический вопрос",
    "2": "анализ данных",
    "3": "взаимодействие с графом",
    "4": "анализ данных и взаимодействие с графом",
    "5": "чушь"
}

CLASS_NAMES = list(DIGIT2CLASS.values())

# ================== PROMPT (подробный, как во втором коде) ==================
def build_prompt(query: str) -> str:
    return f"""Ты — строгий классификатор пользовательских запросов.

Определи один класс:
1 — теоретический вопрос по машинному обучению, математике, статистике, алгоритмам, метрикам, оптимизации, линейной алгебре, вероятностям и смежной теории.
2 — анализ данных: EDA, очистка, визуализация, поиск закономерностей, статистический анализ, сравнение распределений, обработка табличных данных, интерпретация данных.
3 — взаимодействие с графом: построение/редактирование ML-пайплайна, создание узлов, соединение узлов, назначение типа узла, порядок операций в графе, настройка датасета/энкодера/модели/преобразований как элементов графа.
4 — одновременно анализ данных и взаимодействие с графом: запрос содержит и работу с данными, и действия по построению/редактированию графа.
5 — всё остальное: нерелевантные запросы, белиберда, бытовые вопросы, общая болтовня, запросы не про ML и не про перечисленные классы.

Правило приоритета:
- Если есть и анализ данных, и построение/редактирование графа, выбирай 4.
- Иначе если речь о графе, выбирай 3.
- Иначе если речь об анализе данных, выбирай 2.
- Иначе если вопрос теоретический, выбирай 1.
- Иначе выбирай 5.

Отвечай только одним символом: 1, 2, 3, 4 или 5.
Запрос: {query}
Ответ:"""

# ================== DATA ==================
print("=" * 60)
print("ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ")
print("=" * 60)

df_train = pd.read_csv(TRAIN_PATH, sep=';', encoding='windows-1251').dropna()
df_test_raw = pd.read_csv(TEST_PATH, sep=';', encoding='windows-1251').dropna()

# ================== LABEL MAPPING ==================
def label_to_digit(label_str):
    s = str(label_str).strip()

    # уже цифра
    if s in DIGIT2CLASS:
        return s

    # текст → цифра
    inv_map = {v: k for k, v in DIGIT2CLASS.items()}
    return inv_map.get(s, None)

df_train["digit"] = df_train["label"].apply(label_to_digit)
df_test_raw["digit"] = df_test_raw["label"].apply(label_to_digit)

# ================== FILTER ==================
df_train = df_train[df_train["digit"].notna()]
df_test_raw = df_test_raw[df_test_raw["digit"].notna()]

# защита от пустого датасета
assert len(df_test_raw) > 0, "df_test_raw is EMPTY after label mapping"

# ================== SPLIT ==================
test_val, test_final = train_test_split(
    df_test_raw,
    test_size=0.35,
    random_state=42,
    stratify=df_test_raw["digit"]
)

print(f"Train size: {len(df_train)}")
print(f"Validation size: {len(test_val)}")
print(f"Final test size: {len(test_final)}")

# ================== HF DATASET ==================
def df_to_dataset(df):
    return Dataset.from_dict({
        "query": df["query"].tolist(),
        "digit": df["digit"].tolist()
    })

dataset_dict = DatasetDict({
    "train": df_to_dataset(df_train),
    "validation": df_to_dataset(test_val),
    "test": df_to_dataset(test_final)
})

val_queries = dataset_dict["validation"]["query"]
val_digits = dataset_dict["validation"]["digit"]
# ================== TOKENIZER ==================
def tokenize(examples, tokenizer):
    texts = [build_prompt(q) for q in examples["query"]]

    enc = tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

    enc["labels"] = [int(d) - 1 for d in examples["digit"]]  # 0..4
    return enc
# ================== METRICS ==================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "f1": f1}
# ================== TRAIN ==================
def train_llm(model_name):

    print(f"\n=== {model_name} ===")

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # 8-bit QLoRA config
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        device_map='auto',
        num_labels=5,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        #max_memory={0: "0GiB", 1: "40GiB"},
    )

    model.config.use_cache = False
    hidden_size = model.config.hidden_size
    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        task_type="SEQ_CLS",
        target_modules=[
            "q_proj", "k_proj", "v_proj",
            "o_proj", "gate_proj", "up_proj", "down_proj"],
        modules_to_save=["score"])

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # tokenize
    tokenized = dataset_dict.map(
        lambda x: tokenize(x, tokenizer),
        batched=True,
        remove_columns=dataset_dict["train"].column_names
    )

    data_collator = DataCollatorWithPadding(tokenizer)

    args = TrainingArguments(
        output_dir=os.path.join(OUTPUT_DIR, model_name.replace("/", "_")),
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-4,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=4,
        fp16=True,
        optim="adamw_8bit",
        logging_steps=10,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        report_to="none",
        remove_unused_columns=False,
        seed=SEED,
    )


    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(2)],
    )

    print("TRAIN START")
    trainer.train()

    model.save_pretrained(args.output_dir)
    tokenizer.save_pretrained(args.output_dir)

    # Оценка на тестовом наборе
    result = trainer.evaluate(tokenized["test"])
    print("Test result:", result)
    trainer.model = None
    del trainer
    del model
    
    gc.collect()
    torch.cuda.empty_cache()
    
    # Получаем предсказания для confusion matrix
    pred_output = trainer.predict(tokenized["test"])
    logits = pred_output.predictions                 # (N, seq_len, vocab_size)
    true_labels = pred_output.label_ids              # (N,)

    # Находим id токенов для цифр "1".."5" (или используем encode)
    digit_ids = [tokenizer.encode(d, add_special_tokens=False)[0] for d in ["1","2","3","4","5"]]
    last_logits = logits[:, -1, :]                   # логиты последнего токена
    scores = last_logits[:, digit_ids]               # (N, 5)
    pred_labels = np.argmax(scores, axis=-1)   
    # 0..4
    
    # Сохраняем метрики и данные для графиков
    return {
        "result": result,
        "true_labels": true_labels,
        "pred_labels": pred_labels,
        "model_name": model_name,
    }
    
# ================== RUN ==================
all_info = []   # будет хранить словари с result, pred_labels, true_labels, model_name

for m in MODEL_NAMES:
    try:
        info = train_llm(m)
        all_info.append(info)
    except Exception as e:
        print(f"FAILED {m}: {e}")

# Сохраняем только числовые метрики в CSV
results_rows = []
for info in all_info:
    res = info["result"]
    res["model"] = info["model_name"]
    results_rows.append(res)
pd.DataFrame(results_rows).to_csv(RESULTS_CSV, index=False)
print("Results saved to", RESULTS_CSV)

# ================== ГРАФИКИ ==================
if all_info:
    # 1. Barplot сравнения test accuracy
    accs = [info["result"]["eval_accuracy"] for info in all_info]
    models = [info["model_name"] for info in all_info]

    plt.figure(figsize=(14, 8))
    # Сортируем по возрастанию для наглядности
    sorted_idx = np.argsort(accs)
    plt.barh([models[i] for i in sorted_idx], [accs[i] for i in sorted_idx])
    plt.xlabel("Test Accuracy")
    plt.title("Точность моделей после LoRA")
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, "accuracy_comparison.png"), dpi=300)
    plt.close()
    print("Saved accuracy comparison plot.")

    # 2. Confusion matrices для каждой модели
    for info in all_info:
        model_name = info["model_name"]
        true = info["true_labels"]
        pred = info["pred_labels"]
        cm = confusion_matrix(true, pred, labels=list(range(len(CLASS_NAMES))))
        cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)
        cm_norm = np.nan_to_num(cm_norm)

        # Абсолютная матрица
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
        plt.title(f"Confusion Matrix: {model_name}")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, f"cm_{model_name.replace('/','_')}.png"), dpi=300)
        plt.close()

        # Нормализованная матрица
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm_norm, annot=True, fmt=".2%", cmap="YlOrRd",
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
        plt.title(f"Normalized CM: {model_name}")
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, f"cm_norm_{model_name.replace('/','_')}.png"), dpi=300)
        plt.close()

    print(f"All plots saved to {PLOTS_DIR}")

print("DONE")

In [ ]:
FINAL

In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import PeftModel

# ================== НАСТРОЙКИ ==================
MODEL_DIR = "llm_classifier/apple/apple_SimpleSD-4B-instruct"
MAX_LENGTH = 650
USE_8BIT = True

CLASS_NAMES = [
    "теоретический вопрос",
    "анализ данных",
    "взаимодействие с графом",
    "анализ данных и взаимодействие с графом",
    "чушь"
]

# ================== ЗАГРУЗКА МОДЕЛИ ==================
print("Загрузка токенизатора...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("Загрузка базовой модели...")
bnb_config = BitsAndBytesConfig(load_in_8bit=True) if USE_8BIT else None

base_model = AutoModelForSequenceClassification.from_pretrained(
    "apple/SimpleSD-4B-instruct",
    num_labels=5,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
    low_cpu_mem_usage=True
)

base_model.config.pad_token_id = tokenizer.pad_token_id
base_model.config.use_cache = False

print("Загрузка LoRA адаптеров...")
model = PeftModel.from_pretrained(base_model, MODEL_DIR)
model.config.pad_token_id = tokenizer.pad_token_id
model.eval()
print("Модель загружена! Готов к работе.\n")

# ================== ФУНКЦИЯ ПРЕДСКАЗАНИЯ ==================
def predict(texts):
    if isinstance(texts, str):
        texts = [texts]
    encodings = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
        return_attention_mask=True
    )
    device = next(model.parameters()).device
    encodings = {k: v.to(device) for k, v in encodings.items()}
    with torch.no_grad():
        logits = model(**encodings).logits
        preds = torch.argmax(logits, dim=-1).cpu().numpy()
    return [(int(p), CLASS_NAMES[p]) for p in preds]

# ================== ИНТЕРАКТИВНЫЙ РЕЖИМ ==================
print("Введите текст для классификации (нажмите Enter, чтобы завершить):")
while True:
    user_input = input(">>> ").strip()
    if not user_input:
        break
    idx, name = predict(user_input)[0]
    print(f"Класс: {idx} - {name}\n")

Загрузка токенизатора...
Загрузка базовой модели...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[transformers] Qwen3ForSequenceClassification LOAD REPORT from: apple/SimpleSD-4B-instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Загрузка LoRA адаптеров...
Модель загружена! Готов к работе.

Введите текст для классификации (нажмите Enter, чтобы завершить):


>>>  как график сделать


...

>>>  зачем


Класс: 4 - чушь



>>>  зачем она


Класс: 4 - чушь



KeyboardInterrupt: Interrupted by user

In [2]:
import subprocess

def print_gpu_memory_nvidia_smi():
    try:
        result = subprocess.check_output(
            [
                "nvidia-smi",
                "--query-gpu=index,name,memory.total,memory.used,memory.free",
                "--format=csv,nounits,noheader"
            ]
        ).decode("utf-8").strip()

        print("\n[GPU MEMORY - nvidia-smi]")

        for line in result.split("\n"):
            idx, name, total, used, free = [x.strip() for x in line.split(",")]

            total = int(total) / 1024
            used = int(used) / 1024
            free = int(free) / 1024

            print(f"GPU {idx} ({name})")
            print(f"  Total: {total:.2f} GB")
            print(f"  Used:  {used:.2f} GB")
            print(f"  Free:  {free:.2f} GB\n")

    except Exception as e:
        print("nvidia-smi parse error:", e)

In [7]:
print_gpu_memory_nvidia_smi()


[GPU MEMORY - nvidia-smi]
GPU 0 (NVIDIA RTX 6000 Ada Generation)
  Total: 47.99 GB
  Used:  44.99 GB
  Free:  2.38 GB

GPU 1 (NVIDIA RTX 6000 Ada Generation)
  Total: 47.99 GB
  Used:  47.35 GB
  Free:  0.02 GB



In [ ]:
import shutil
import os
from IPython.display import FileLink

# Укажите путь к папке, которую нужно скачать
folder_path = "llm_plots"  # измените на вашу папку
zip_name = "llm_plots_finetune.zip"

# Создаём ZIP-архив
shutil.make_archive(zip_name.replace('.zip', ''), 'zip', folder_path)

# Отображаем ссылку для скачивания
display(FileLink(zip_name))

In [88]:
Bigger RAM needed below

SyntaxError: invalid syntax (3987417878.py, line 1)

In [ ]:
Обучение Gemma

In [1]:
"""
Обучение LLM с помощью QLoRA для классификации запросов.
Исправленная версия: стабильный 8-bit LoRA + корректный Trainer + O(N) evaluation.
Добавлены детальный промпт и графики (сравнение accuracy, confusion matrices).
"""

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import gc
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    BitsAndBytesConfig,
    EarlyStoppingCallback,
    AutoModelForSequenceClassification
)

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

# ================== НАСТРОЙКИ ==================
TRAIN_PATH = "train.csv"
TEST_PATH = "test.csv"

MAX_LENGTH = 620
OUTPUT_DIR = "./llm_classifier"
PLOTS_DIR = "./llm_plots"
RESULTS_CSV = "results_normal.csv"

os.makedirs(PLOTS_DIR, exist_ok=True)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

MODEL_NAMES = [
    "mistralai/Mistral-7B-Instruct-v0.3",
    "apple/SimpleSD-4B-instruct",
    "Qwen/Qwen3-4B-Instruct-2507",
    "Qwen/Qwen2.5-7B-Instruct",
    "unsloth/Meta-Llama-3.1-8B-Instruct",
    "openchat/openchat-3.6-8b-20240522",
    "AvitoTech/avibe",
    "ai-sage/GigaChat3.1-10B-A1.8B-bf16"
]

DIGIT2CLASS = {
    "1": "теоретический вопрос",
    "2": "анализ данных",
    "3": "взаимодействие с графом",
    "4": "анализ данных и взаимодействие с графом",
    "5": "чушь"
}

CLASS_NAMES = list(DIGIT2CLASS.values())
# ================== DATA ==================
print("=" * 60)
print("ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ")
print("=" * 60)

df_train = pd.read_csv(TRAIN_PATH, sep=';', encoding='windows-1251').dropna()
df_test_raw = pd.read_csv(TEST_PATH, sep=';', encoding='windows-1251').dropna()

# ================== LABEL MAPPING ==================
def label_to_digit(label_str):
    s = str(label_str).strip()

    # уже цифра
    if s in DIGIT2CLASS:
        return s

    # текст → цифра
    inv_map = {v: k for k, v in DIGIT2CLASS.items()}
    return inv_map.get(s, None)

df_train["digit"] = df_train["label"].apply(label_to_digit)
df_test_raw["digit"] = df_test_raw["label"].apply(label_to_digit)

# ================== FILTER ==================
df_train = df_train[df_train["digit"].notna()]
df_test_raw = df_test_raw[df_test_raw["digit"].notna()]

# защита от пустого датасета
assert len(df_test_raw) > 0, "df_test_raw is EMPTY after label mapping"

# ================== SPLIT ==================
test_val, test_final = train_test_split(
    df_test_raw,
    test_size=0.65,
    random_state=42,
    stratify=df_test_raw["digit"]
)

print(f"Train size: {len(df_train)}")
print(f"Validation size: {len(test_val)}")
print(f"Final test size: {len(test_final)}")

# ================== DATASET ==================
def df_to_dataset(df):
    return Dataset.from_dict({
        "query": df["query"].tolist(),
        "digit": df["digit"].tolist()
    })

dataset_dict = DatasetDict({
    "train": df_to_dataset(df_train),
    "validation": df_to_dataset(test_val),
    "test": df_to_dataset(test_final)
})

val_queries = dataset_dict["validation"]["query"]
val_digits = dataset_dict["validation"]["digit"]
# ================== TOKENIZER ==================
def tokenize(examples, tokenizer):
    enc = tokenizer(
        examples["query"],
        truncation=True,
        #padding="max_length",
        max_length=MAX_LENGTH,
    )
    enc["labels"] = [int(d) - 1 for d in examples["digit"]]  # 0..4
    return enc
# ================== METRICS ==================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "f1": f1}


print("Train distribution:")
print(df_train["digit"].value_counts(normalize=True))

print("Validation distribution:")
print(test_val["digit"].value_counts(normalize=True))
# ================== TRAIN ==================
def train_llm(model_name):

    print(f"\n=== {model_name} ===")

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    #8-bit QLoRA config
    # bnb_config = BitsAndBytesConfig(
    #     load_in_8bit=True
    # )

    model_kwargs = {
        "device_map": "auto",
        "num_labels": 5,
        "torch_dtype": torch.bfloat16,
        "trust_remote_code": True,
        "low_cpu_mem_usage": True,
    }

    if "GigaChat" not in model_name:
        model_kwargs["attn_implementation"] = "sdpa" # <-- Магия происходит здесь
    else:
        print(f"⚠️ Для {model_name} оптимизация отключена во избежание конфликтов.")    

    
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        #device_map='auto',
        #num_labels=5,
        #quantization_config=bnb_config,
        #torch_dtype=torch.bfloat16,
        #trust_remote_code=True,
        #low_cpu_mem_usage=True,
        **model_kwargs
    )
    
    
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False
    #model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.035,
        bias="none",
        task_type="SEQ_CLS",
        target_modules="all-linear")
    
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # tokenize
    tokenized = dataset_dict.map(
        lambda x: tokenize(x, tokenizer),
        batched=True,
        remove_columns=dataset_dict["train"].column_names
    )

    data_collator = DataCollatorWithPadding(tokenizer)

    args = TrainingArguments(
        output_dir=os.path.join(OUTPUT_DIR, model_name.replace("/", "_")),
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=2,
        num_train_epochs=5,
        #fp16=True,
        bf16=True,
        optim="adamw_8bit",
        logging_steps=10,
        save_total_limit=3,
        metric_for_best_model="f1",
        load_best_model_at_end=True,
        greater_is_better=True,
        report_to="none",
        warmup_ratio=0.12,
        remove_unused_columns=False,
        seed=SEED,
    )


    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(2)],)

    print("TRAIN START")
    trainer.train()

    model.save_pretrained(args.output_dir)
    tokenizer.save_pretrained(args.output_dir)

    # Оценка на тестовом наборе
    result = trainer.evaluate(tokenized["test"])
    print("Test result:", result)
    
    # Предсказания
    pred_output = trainer.predict(tokenized["test"])
    
    logits = pred_output.predictions      # (N, 5)
    true_labels = pred_output.label_ids   # (N,)
    pred_labels = np.argmax(logits, axis=1)
    
    # очистка памяти
    trainer.model = None
    del trainer
    del model
    torch.cuda.synchronize()
    gc.collect()
    torch.cuda.empty_cache()
    
    return {
        "result": result,
        "true_labels": true_labels,
        "pred_labels": pred_labels,
        "model_name": model_name,
    }    
# ================== RUN ==================
all_info = []   # будет хранить словари с result, pred_labels, true_labels, model_name

for m in MODEL_NAMES:
    try:
        info = train_llm(m)
        all_info.append(info)
    except Exception as e:
        print(f"FAILED {m}: {e}")

# Сохраняем только числовые метрики в CSV
results_rows = []
for info in all_info:
    res = info["result"]
    res["model"] = info["model_name"]
    results_rows.append(res)
pd.DataFrame(results_rows).to_csv(RESULTS_CSV, index=False)
print("Results saved to", RESULTS_CSV)

# ================== ГРАФИКИ ==================
if all_info:
    # 1. График сравнения точности моделей
    plt.figure(figsize=(14, 8))

    # Подготовка данных
    models = [info["model_name"] for info in all_info]
    accs = [info["result"]["eval_accuracy"] for info in all_info]
    f1s = [info["result"]["eval_f1"] for info in all_info]

    comparison_df = pd.DataFrame({
        'Model': models,
        'Accuracy': accs,
        'F1-Score': f1s
    })

    # Сортируем модели по точности
    sorted_models = comparison_df.sort_values('Accuracy', ascending=True)

    # Создаем столбчатую диаграмму с двумя метриками
    x = range(len(sorted_models))
    width = 0.35

    plt.barh([i - width/2 for i in x], sorted_models['Accuracy'],
            width, label='Accuracy', color='skyblue', edgecolor='navy')
    plt.barh([i + width/2 for i in x], sorted_models['F1-Score'],
            width, label='F1-Score', color='lightcoral', edgecolor='darkred')

    # Добавляем значения на столбцы
    for i, (acc, f1) in enumerate(zip(sorted_models['Accuracy'], sorted_models['F1-Score'])):
        plt.text(acc + 0.01, i - width/2, f'{acc:.3f}',
                va='center', fontsize=9, fontweight='bold', color='navy')
        plt.text(f1 + 0.01, i + width/2, f'{f1:.3f}',
                va='center', fontsize=9, fontweight='bold', color='darkred')

    plt.yticks(x, sorted_models['Model'])
    plt.xlabel('Значение метрики', fontsize=12)
    plt.title('Сравнение производительности моделей',
             fontsize=16, fontweight='bold', pad=20)
    plt.legend(loc='lower right')
    plt.grid(axis='x', alpha=0.3)
    plt.xlim([0, 1.1])

    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, "models_comparison_normal.png"), dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ График сравнения сохранен: {PLOTS_DIR}/models_comparison.png")

    # 2. Confusion matrices для каждой модели
    for info in all_info:
        model_name = info["model_name"]
        true = info["true_labels"]
        pred = info["pred_labels"]
        cm = confusion_matrix(true, pred, labels=list(range(len(CLASS_NAMES))))
        cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)
        cm_norm = np.nan_to_num(cm_norm)

        # Абсолютная матрица
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
        plt.title(f"Confusion Matrix: {model_name}")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, f"cm_{model_name.replace('/','_')}.png"), dpi=300)
        plt.close()

        # Нормализованная матрица
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm_norm, annot=True, fmt=".2%", cmap="YlOrRd",
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
        plt.title(f"Normalized CM: {model_name}")
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, f"cm_norm_{model_name.replace('/','_')}.png"), dpi=300)
        plt.close()

    print(f"All plots saved to {PLOTS_DIR}")

print("DONE")

ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ
Train size: 1608
Validation size: 237
Final test size: 442
Train distribution:
digit
3    0.226368
1    0.199005
5    0.193408
2    0.190920
4    0.190299
Name: proportion, dtype: float64
Validation distribution:
digit
5    0.261603
1    0.240506
3    0.172996
4    0.168776
2    0.156118
Name: proportion, dtype: float64

=== mistralai/Mistral-7B-Instruct-v0.3 ===


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] MistralForSequenceClassification LOAD REPORT from: mistralai/Mistral-7B-Instruct-v0.3
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 20,992,000 || all params: 7,134,818,304 || trainable%: 0.2942


...

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.260726,0.277820,5,0.943439,0.941053


Test result: {'eval_loss': 0.27782002091407776, 'eval_accuracy': 0.9434389140271493, 'eval_f1': 0.9410531975337314}



=== ai-sage/GigaChat3.1-10B-A1.8B-bf16 ===


config.json: 0.00B [00:00, ?B/s]

FAILED ai-sage/GigaChat3.1-10B-A1.8B-bf16: Validation error for field 'routed_scaling_factor':
    TypeError: Field 'routed_scaling_factor' expected float, got int (value: 1)
Results saved to results_normal.csv
✅ График сравнения сохранен: ./llm_plots/models_comparison.png
All plots saved to ./llm_plots
DONE


In [ ]:
тест последний

In [1]:
"""
Обучение LLM с помощью QLoRA для классификации запросов.
Исправленная версия: стабильный 8-bit LoRA + корректный Trainer + O(N) evaluation.
Добавлены детальный промпт и графики (сравнение accuracy, confusion matrices).
"""

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import gc
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    BitsAndBytesConfig,
    EarlyStoppingCallback,
    AutoModelForSequenceClassification
)

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

# ================== НАСТРОЙКИ ==================
TRAIN_PATH = "train.csv"
TEST_PATH = "test.csv"

MAX_LENGTH = 620
OUTPUT_DIR = "./llm_classifier"
PLOTS_DIR = "./llm_plots"
RESULTS_CSV = "results_normal.csv"

os.makedirs(PLOTS_DIR, exist_ok=True)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

MODEL_NAMES = [
    'mistralai/Mistral-7B-Instruct-v0.3',
    "apple/SimpleSD-4B-instruct",
    "Qwen/Qwen3-4B-Instruct-2507",
    "Qwen/Qwen2.5-7B-Instruct",
    "unsloth/Meta-Llama-3.1-8B-Instruct",
    "openchat/openchat-3.6-8b-20240522",
    "google/gemma-4-E2B-it",
    "google/gemma-4-E4B-it",
    "AvitoTech/avibe",
    "OpenPipe/Qwen3-14B-Instruct",
    "unsloth/Mistral-Nemo-Instruct-2407",
    'ai-sage/GigaChat3.1-10B-A1.8B-bf16',
    'yandex/YandexGPT-5-Lite-8B-instruct'
]

DIGIT2CLASS = {
    "1": "теоретический вопрос",
    "2": "анализ данных",
    "3": "взаимодействие с графом",
    "4": "анализ данных и взаимодействие с графом",
    "5": "чушь"
}

CLASS_NAMES = list(DIGIT2CLASS.values())
# ================== DATA ==================
print("=" * 60)
print("ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ")
print("=" * 60)

df_train = pd.read_csv(TRAIN_PATH, sep=';', encoding='windows-1251').dropna()
df_test_raw = pd.read_csv(TEST_PATH, sep=';', encoding='windows-1251').dropna()

# ================== LABEL MAPPING ==================
def label_to_digit(label_str):
    s = str(label_str).strip()

    # уже цифра
    if s in DIGIT2CLASS:
        return s

    # текст → цифра
    inv_map = {v: k for k, v in DIGIT2CLASS.items()}
    return inv_map.get(s, None)

df_train["digit"] = df_train["label"].apply(label_to_digit)
df_test_raw["digit"] = df_test_raw["label"].apply(label_to_digit)

# ================== FILTER ==================
df_train = df_train[df_train["digit"].notna()]
df_test_raw = df_test_raw[df_test_raw["digit"].notna()]

# защита от пустого датасета
assert len(df_test_raw) > 0, "df_test_raw is EMPTY after label mapping"

# ================== SPLIT ==================
test_val, test_final = train_test_split(
    df_test_raw,
    test_size=0.65,
    random_state=42,
    stratify=df_test_raw["digit"]
)

print(f"Train size: {len(df_train)}")
print(f"Validation size: {len(test_val)}")
print(f"Final test size: {len(test_final)}")

# ================== DATASET ==================
def df_to_dataset(df):
    return Dataset.from_dict({
        "query": df["query"].tolist(),
        "digit": df["digit"].tolist()
    })

dataset_dict = DatasetDict({
    "train": df_to_dataset(df_train),
    "validation": df_to_dataset(test_val),
    "test": df_to_dataset(test_final)
})

val_queries = dataset_dict["validation"]["query"]
val_digits = dataset_dict["validation"]["digit"]
# ================== TOKENIZER ==================
def tokenize(examples, tokenizer):
    enc = tokenizer(
        examples["query"],
        truncation=True,
        #padding="max_length",
        max_length=MAX_LENGTH,
    )
    enc["labels"] = [int(d) - 1 for d in examples["digit"]]  # 0..4
    return enc
# ================== METRICS ==================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "f1": f1}


print("Train distribution:")
print(df_train["digit"].value_counts(normalize=True))

print("Validation distribution:")
print(test_val["digit"].value_counts(normalize=True))
# ================== TRAIN ==================
def train_llm(model_name):

    print(f"\n=== {model_name} ===")

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    #8-bit QLoRA config
    # bnb_config = BitsAndBytesConfig(
    #     load_in_8bit=True
    # )

    model_kwargs = {
        "device_map": "auto",
        "num_labels": 5,
        "torch_dtype": torch.bfloat16,
        "trust_remote_code": True,
        "low_cpu_mem_usage": True,
    }

    if "GigaChat" not in model_name:
        model_kwargs["attn_implementation"] = "sdpa" # <-- Магия происходит здесь
    else:
        print(f"⚠️ Для {model_name} оптимизация отключена во избежание конфликтов.")    

    
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        #device_map='auto',
        #num_labels=5,
        #quantization_config=bnb_config,
        #torch_dtype=torch.bfloat16,
        #trust_remote_code=True,
        #low_cpu_mem_usage=True,
        **model_kwargs
    )
    
    
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False
    #model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.035,
        bias="none",
        task_type="SEQ_CLS",
        target_modules="all-linear")
    
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # tokenize
    tokenized = dataset_dict.map(
        lambda x: tokenize(x, tokenizer),
        batched=True,
        remove_columns=dataset_dict["train"].column_names
    )

    data_collator = DataCollatorWithPadding(tokenizer)

    args = TrainingArguments(
        output_dir=os.path.join(OUTPUT_DIR, model_name.replace("/", "_")),
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=2,
        num_train_epochs=5,
        #fp16=True,
        bf16=True,
        optim="adamw_8bit",
        logging_steps=10,
        save_total_limit=3,
        metric_for_best_model="f1",
        load_best_model_at_end=True,
        greater_is_better=True,
        report_to="none",
        warmup_ratio=0.12,
        remove_unused_columns=False,
        seed=SEED,
    )


    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(2)],)

    print("TRAIN START")
    trainer.train()

    model.save_pretrained(args.output_dir)
    tokenizer.save_pretrained(args.output_dir)

    # Оценка на тестовом наборе
    result = trainer.evaluate(tokenized["test"])
    print("Test result:", result)
    
    # Предсказания
    pred_output = trainer.predict(tokenized["test"])
    
    logits = pred_output.predictions      # (N, 5)
    true_labels = pred_output.label_ids   # (N,)
    pred_labels = np.argmax(logits, axis=1)
    
    # очистка памяти
    trainer.model = None
    del trainer
    del model
    torch.cuda.synchronize()
    gc.collect()
    torch.cuda.empty_cache()
    
    return {
        "result": result,
        "true_labels": true_labels,
        "pred_labels": pred_labels,
        "model_name": model_name,
    }    
# ================== RUN ==================
all_info = []   # будет хранить словари с result, pred_labels, true_labels, model_name

for m in MODEL_NAMES:
    try:
        info = train_llm(m)
        all_info.append(info)
    except Exception as e:
        print(f"FAILED {m}: {e}")

# Сохраняем только числовые метрики в CSV
results_rows = []
for info in all_info:
    res = info["result"]
    res["model"] = info["model_name"]
    results_rows.append(res)
pd.DataFrame(results_rows).to_csv(RESULTS_CSV, index=False)
print("Results saved to", RESULTS_CSV)

# ================== ГРАФИКИ ==================
if all_info:
    # 1. График сравнения точности моделей
    plt.figure(figsize=(14, 8))

    # Подготовка данных
    models = [info["model_name"] for info in all_info]
    accs = [info["result"]["eval_accuracy"] for info in all_info]
    f1s = [info["result"]["eval_f1"] for info in all_info]

    comparison_df = pd.DataFrame({
        'Model': models,
        'Accuracy': accs,
        'F1-Score': f1s
    })

    # Сортируем модели по точности
    sorted_models = comparison_df.sort_values('Accuracy', ascending=True)

    # Создаем столбчатую диаграмму с двумя метриками
    x = range(len(sorted_models))
    width = 0.35

    plt.barh([i - width/2 for i in x], sorted_models['Accuracy'],
            width, label='Accuracy', color='skyblue', edgecolor='navy')
    plt.barh([i + width/2 for i in x], sorted_models['F1-Score'],
            width, label='F1-Score', color='lightcoral', edgecolor='darkred')

    # Добавляем значения на столбцы
    for i, (acc, f1) in enumerate(zip(sorted_models['Accuracy'], sorted_models['F1-Score'])):
        plt.text(acc + 0.01, i - width/2, f'{acc:.3f}',
                va='center', fontsize=9, fontweight='bold', color='navy')
        plt.text(f1 + 0.01, i + width/2, f'{f1:.3f}',
                va='center', fontsize=9, fontweight='bold', color='darkred')

    plt.yticks(x, sorted_models['Model'])
    plt.xlabel('Значение метрики', fontsize=12)
    plt.title('Сравнение производительности моделей',
             fontsize=16, fontweight='bold', pad=20)
    plt.legend(loc='lower right')
    plt.grid(axis='x', alpha=0.3)
    plt.xlim([0, 1.1])

    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, "models_comparison_normal.png"), dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ График сравнения сохранен: {PLOTS_DIR}/models_comparison.png")

    # 2. Confusion matrices для каждой модели
    for info in all_info:
        model_name = info["model_name"]
        true = info["true_labels"]
        pred = info["pred_labels"]
        cm = confusion_matrix(true, pred, labels=list(range(len(CLASS_NAMES))))
        cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)
        cm_norm = np.nan_to_num(cm_norm)

        # Абсолютная матрица
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
        plt.title(f"Confusion Matrix: {model_name}")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, f"cm_{model_name.replace('/','_')}.png"), dpi=300)
        plt.close()

        # Нормализованная матрица
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm_norm, annot=True, fmt=".2%", cmap="YlOrRd",
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
        plt.title(f"Normalized CM: {model_name}")
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, f"cm_norm_{model_name.replace('/','_')}.png"), dpi=300)
        plt.close()

    print(f"All plots saved to {PLOTS_DIR}")

print("DONE")

ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ
Train size: 1608
Validation size: 237
Final test size: 442
Train distribution:
digit
3    0.226368
1    0.199005
5    0.193408
2    0.190920
4    0.190299
Name: proportion, dtype: float64
Validation distribution:
digit
5    0.261603
1    0.240506
3    0.172996
4    0.168776
2    0.156118
Name: proportion, dtype: float64

=== mistralai/Mistral-7B-Instruct-v0.3 ===


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] MistralForSequenceClassification LOAD REPORT from: mistralai/Mistral-7B-Instruct-v0.3
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 20,992,000 || all params: 7,134,818,304 || trainable%: 0.2942


...

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.012684,0.473153,5,0.916290,0.914210


Test result: {'eval_loss': 0.473153293132782, 'eval_accuracy': 0.916289592760181, 'eval_f1': 0.9142101404121865}



=== ai-sage/GigaChat3.1-10B-A1.8B-bf16 ===
FAILED ai-sage/GigaChat3.1-10B-A1.8B-bf16: Validation error for field 'routed_scaling_factor':
    TypeError: Field 'routed_scaling_factor' expected float, got int (value: 1)

=== yandex/YandexGPT-5-Lite-8B-instruct ===


[transformers] Could not extract SentencePiece model from /home/user-17/.cache/huggingface/hub/models--yandex--YandexGPT-5-Lite-8B-instruct/snapshots/b556811768376b46c69caab60c4d1b69df9faaa1/tokenizer.model using sentencepiece library due to 
SentencePieceExtractor requires the protobuf library but it was not found in your environment. Check out the instructions on the
installation page of its repo: https://github.com/protocolbuffers/protobuf/tree/master/python#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.
. Falling back to TikToken extractor.


FAILED yandex/YandexGPT-5-Lite-8B-instruct: Error parsing line b'\x0e' in /home/user-17/.cache/huggingface/hub/models--yandex--YandexGPT-5-Lite-8B-instruct/snapshots/b556811768376b46c69caab60c4d1b69df9faaa1/tokenizer.model
Results saved to results_normal.csv
✅ График сравнения сохранен: ./llm_plots/models_comparison.png
All plots saved to ./llm_plots
DONE


In [ ]:
yandex/gigichat finetune

In [1]:

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import gc
import time
import numpy as np
from transformers import AutoConfig, AutoModelForCausalLM 
from huggingface_hub import hf_hub_download
import json
import pandas as pd
import torch
from transformers import AutoConfig
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    BitsAndBytesConfig,
    EarlyStoppingCallback,
    AutoModelForSequenceClassification
)

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

# ================== НАСТРОЙКИ ==================
TRAIN_PATH = "train.csv"
TEST_PATH = "test.csv"

MAX_LENGTH = 620
OUTPUT_DIR = "./llm_classifier"
PLOTS_DIR = "./llm_plots"
RESULTS_CSV = "results_rusmodels.csv"

os.makedirs(PLOTS_DIR, exist_ok=True)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

MODEL_NAMES = [
    # 'mistralai/Mistral-7B-Instruct-v0.3',
    # "apple/SimpleSD-4B-instruct",
    # "Qwen/Qwen3-4B-Instruct-2507",
    # "Qwen/Qwen2.5-7B-Instruct",
    # "unsloth/Meta-Llama-3.1-8B-Instruct",
    # "openchat/openchat-3.6-8b-20240522",
    # "google/gemma-4-E2B-it",
    # "google/gemma-4-E4B-it",
    # "AvitoTech/avibe",
    # "OpenPipe/Qwen3-14B-Instruct",
    # "unsloth/Mistral-Nemo-Instruct-2407",
    'ai-sage/GigaChat3.1-10B-A1.8B-bf16',
    'yandex/YandexGPT-5-Lite-8B-instruct'
]

DIGIT2CLASS = {
    "1": "теоретический вопрос",
    "2": "анализ данных",
    "3": "взаимодействие с графом",
    "4": "анализ данных и взаимодействие с графом",
    "5": "чушь"
}

CLASS_NAMES = list(DIGIT2CLASS.values())
# ================== DATA ==================
print("=" * 60)
print("ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ")
print("=" * 60)

df_train = pd.read_csv(TRAIN_PATH, sep=';', encoding='windows-1251').dropna()
df_test_raw = pd.read_csv(TEST_PATH, sep=';', encoding='windows-1251').dropna()

# ================== LABEL MAPPING ==================
def label_to_digit(label_str):
    s = str(label_str).strip()

    # уже цифра
    if s in DIGIT2CLASS:
        return s

    # текст → цифра
    inv_map = {v: k for k, v in DIGIT2CLASS.items()}
    return inv_map.get(s, None)

df_train["digit"] = df_train["label"].apply(label_to_digit)
df_test_raw["digit"] = df_test_raw["label"].apply(label_to_digit)

# ================== FILTER ==================
df_train = df_train[df_train["digit"].notna()]
df_test_raw = df_test_raw[df_test_raw["digit"].notna()]

# защита от пустого датасета
assert len(df_test_raw) > 0, "df_test_raw is EMPTY after label mapping"

# ================== SPLIT ==================
test_val, test_final = train_test_split(
    df_test_raw,
    test_size=0.65,
    random_state=42,
    stratify=df_test_raw["digit"]
)

print(f"Train size: {len(df_train)}")
print(f"Validation size: {len(test_val)}")
print(f"Final test size: {len(test_final)}")

# ================== DATASET ==================
def df_to_dataset(df):
    return Dataset.from_dict({
        "query": df["query"].tolist(),
        "digit": df["digit"].tolist()
    })

dataset_dict = DatasetDict({
    "train": df_to_dataset(df_train),
    "validation": df_to_dataset(test_val),
    "test": df_to_dataset(test_final)
})

val_queries = dataset_dict["validation"]["query"]
val_digits = dataset_dict["validation"]["digit"]
# ================== TOKENIZER ==================
def tokenize(examples, tokenizer):
    enc = tokenizer(
        examples["query"],
        truncation=True,
        #padding="max_length",
        max_length=MAX_LENGTH,
    )
    enc["labels"] = [int(d) - 1 for d in examples["digit"]]  # 0..4
    return enc
# ================== METRICS ==================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "f1": f1}


print("Train distribution:")
print(df_train["digit"].value_counts(normalize=True))

print("Validation distribution:")
print(test_val["digit"].value_counts(normalize=True))


from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM
from transformers.modeling_outputs import SequenceClassifierOutput
from huggingface_hub import hf_hub_download
import json
import torch
from peft import LoraConfig, get_peft_model

def patch_peft_sequence_classification(model, tokenizer):
    """
    Универсальный обезвреживатель кастомных моделей (YandexGPT, GigaChat).
    Исправлена ошибка булевого значения тензора лосса за счет принудительного .squeeze()
    """
    old_forward = model.forward

    def safe_forward(*args, **kwargs):
        labels = kwargs.pop("labels", None)
        
        outputs = old_forward(*args, **kwargs)
        logits = outputs.logits if hasattr(outputs, "logits") else outputs[0]
        
        # Пулинг трехмерного тензора (B, S, C) -> (B, C)
        if len(logits.shape) == 3:
            attention_mask = kwargs.get("attention_mask")
            input_ids = kwargs.get("input_ids") or (args[0] if len(args) > 0 else None)
            
            if attention_mask is None and input_ids is not None:
                attention_mask = input_ids.ne(tokenizer.pad_token_id).long()
                
            if attention_mask is not None:
                sequence_lengths = attention_mask.sum(dim=-1) - 1
                sequence_lengths = torch.clamp(sequence_lengths, min=0)
                pooled_logits = logits[torch.arange(logits.shape[0], device=logits.device), sequence_lengths]
            else:
                pooled_logits = logits[:, -1, :]
        else:
            pooled_logits = logits

        # Расчет CrossEntropyLoss
        loss = None
        if labels is not None:
            loss_fct = torch.nn.CrossEntropyLoss()
            # Насильно превращаем лосс в 0-мерный скаляр (.squeeze()), чтобы Trainer не ругался
            loss = loss_fct(pooled_logits.view(-1, model.config.num_labels), labels.view(-1)).squeeze()
            
        return SequenceClassifierOutput(
            loss=loss,
            logits=pooled_logits,
            hidden_states=getattr(outputs, "hidden_states", None),
            attentions=getattr(outputs, "attentions", None)
        )

    model.forward = safe_forward


# ================== TRAIN ==================
def train_llm(model_name):
    print(f"\n=== {model_name} ===")

    # 1. Патч config.json (для GigaChat)
    try:
        config_file = hf_hub_download(repo_id=model_name, filename="config.json")
        with open(config_file, "r") as f:
            config_dict = json.load(f)
        if "routed_scaling_factor" in config_dict and isinstance(config_dict["routed_scaling_factor"], int):
            config_dict["routed_scaling_factor"] = float(config_dict["routed_scaling_factor"])
            with open(config_file, "w") as f:
                json.dump(config_dict, f, indent=2)
            print(f"[{model_name}] Пропатчили 'routed_scaling_factor' в локальном кэше config.json.")
    except Exception as e:
        print(f"[{model_name}] Пропустили патч кэша: {e}")

    config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)

    # 2. Токенизатор
    tokenizer = AutoTokenizer.from_pretrained(model_name, config=config, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token or tokenizer.unk_token
    tokenizer.padding_side = "right"

    # 3. model_kwargs
    model_kwargs = {
        "device_map": "auto",
        "torch_dtype": torch.bfloat16,
        "trust_remote_code": True,
        "low_cpu_mem_usage": True,
        "config": config,
    }

    if "GigaChat" not in model_name:
        model_kwargs["attn_implementation"] = "sdpa"
    else:
        print(f"⚠️ Для {model_name} оптимизация отключена во избежание конфликтов.")    

    # 4. Загрузка базовой модели
    base_model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    base_model.config.pad_token_id = tokenizer.pad_token_id
    base_model.config.num_labels = 5
    base_model.config.use_cache = False

    # 5. Обертка в PEFT (Явный список модулей вместо "all-linear")
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.035,
        bias="none",
        task_type="SEQ_CLS",
        target_modules=["q_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    )
    
    model = get_peft_model(base_model, lora_config)
    
    # Применяем исправленный патч пулинга и лосса
    patch_peft_sequence_classification(model, tokenizer)
    
    model.print_trainable_parameters()
    
    # tokenize
    tokenized = dataset_dict.map(
        lambda x: tokenize(x, tokenizer),
        batched=True,
        remove_columns=dataset_dict["train"].column_names
    )

    data_collator = DataCollatorWithPadding(tokenizer)

    args = TrainingArguments(
        output_dir=os.path.join(OUTPUT_DIR, model_name.replace("/", "_")),
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=2,
        num_train_epochs=5,
        bf16=True,
        optim="adamw_8bit",
        logging_steps=10,
        save_total_limit=3,
        metric_for_best_model="f1",
        load_best_model_at_end=True,
        greater_is_better=True,
        report_to="none",
        warmup_ratio=0.12,
        remove_unused_columns=False,
        seed=SEED,
    )

    # Используем стандартный Trainer, наш патч внутри модели сделает всю работу за него!
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(2)],
    )

    print("TRAIN START")
    trainer.train()

    model.save_pretrained(args.output_dir)
    tokenizer.save_pretrained(args.output_dir)

    result = trainer.evaluate(tokenized["test"])
    print("Test result:", result)
    
    pred_output = trainer.predict(tokenized["test"])
    
    logits = pred_output.predictions      
    true_labels = pred_output.label_ids   
    pred_labels = np.argmax(logits, axis=1)
    
    trainer.model = None
    del trainer
    del model
    torch.cuda.synchronize()
    gc.collect()
    torch.cuda.empty_cache()
    
    return {
        "result": result,
        "true_labels": true_labels,
        "pred_labels": pred_labels,
        "model_name": model_name,
    }
# ================== RUN ==================
all_info = []   # будет хранить словари с result, pred_labels, true_labels, model_name

for m in MODEL_NAMES:
    try:
        info = train_llm(m)
        all_info.append(info)
    except Exception as e:
        print(f"FAILED {m}: {e}")

# Сохраняем только числовые метрики в CSV
results_rows = []
for info in all_info:
    res = info["result"]
    res["model"] = info["model_name"]
    results_rows.append(res)
pd.DataFrame(results_rows).to_csv(RESULTS_CSV, index=False)
print("Results saved to", RESULTS_CSV)

# ================== ГРАФИКИ ==================
if all_info:
    # 1. График сравнения точности моделей
    plt.figure(figsize=(14, 8))

    # Подготовка данных
    models = [info["model_name"] for info in all_info]
    accs = [info["result"]["eval_accuracy"] for info in all_info]
    f1s = [info["result"]["eval_f1"] for info in all_info]

    comparison_df = pd.DataFrame({
        'Model': models,
        'Accuracy': accs,
        'F1-Score': f1s
    })

    # Сортируем модели по точности
    sorted_models = comparison_df.sort_values('Accuracy', ascending=True)

    # Создаем столбчатую диаграмму с двумя метриками
    x = range(len(sorted_models))
    width = 0.35

    plt.barh([i - width/2 for i in x], sorted_models['Accuracy'],
            width, label='Accuracy', color='skyblue', edgecolor='navy')
    plt.barh([i + width/2 for i in x], sorted_models['F1-Score'],
            width, label='F1-Score', color='lightcoral', edgecolor='darkred')

    # Добавляем значения на столбцы
    for i, (acc, f1) in enumerate(zip(sorted_models['Accuracy'], sorted_models['F1-Score'])):
        plt.text(acc + 0.01, i - width/2, f'{acc:.3f}',
                va='center', fontsize=9, fontweight='bold', color='navy')
        plt.text(f1 + 0.01, i + width/2, f'{f1:.3f}',
                va='center', fontsize=9, fontweight='bold', color='darkred')

    plt.yticks(x, sorted_models['Model'])
    plt.xlabel('Значение метрики', fontsize=12)
    plt.title('Сравнение производительности моделей',
             fontsize=16, fontweight='bold', pad=20)
    plt.legend(loc='lower right')
    plt.grid(axis='x', alpha=0.3)
    plt.xlim([0, 1.1])

    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, "models_comparison_normal.png"), dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ График сравнения сохранен: {PLOTS_DIR}/models_comparison.png")

    # 2. Confusion matrices для каждой модели
    for info in all_info:
        model_name = info["model_name"]
        true = info["true_labels"]
        pred = info["pred_labels"]
        cm = confusion_matrix(true, pred, labels=list(range(len(CLASS_NAMES))))
        cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)
        cm_norm = np.nan_to_num(cm_norm)

        # Абсолютная матрица
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
        plt.title(f"Confusion Matrix: {model_name}")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, f"cm_{model_name.replace('/','_')}.png"), dpi=300)
        plt.close()

        # Нормализованная матрица
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm_norm, annot=True, fmt=".2%", cmap="YlOrRd",
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
        plt.title(f"Normalized CM: {model_name}")
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, f"cm_norm_{model_name.replace('/','_')}.png"), dpi=300)
        plt.close()

    print(f"All plots saved to {PLOTS_DIR}")

print("DONE")

ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ
Train size: 1608
Validation size: 237
Final test size: 442
Train distribution:
digit
3    0.226368
1    0.199005
5    0.193408
2    0.190920
4    0.190299
Name: proportion, dtype: float64
Validation distribution:
digit
5    0.261603
1    0.240506
3    0.172996
4    0.168776
2    0.156118
Name: proportion, dtype: float64

=== ai-sage/GigaChat3.1-10B-A1.8B-bf16 ===


[ai-sage/GigaChat3.1-10B-A1.8B-bf16] Пропатчили 'routed_scaling_factor' в локальном кэше config.json.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/10.7M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/276 [00:00<?, ?B/s]

...

Map:   0%|          | 0/442 [00:00<?, ? examples/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


TRAIN START


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


FAILED yandex/YandexGPT-5-Lite-8B-instruct: Boolean value of Tensor with more than one value is ambiguous
Results saved to results_rusmodels.csv
DONE


In [4]:
import shutil
import os
from IPython.display import FileLink

# Укажите путь к папке, которую нужно скачать
folder_path = "llm_classifier/AvitoTech_avibe"  # измените на вашу папку
zip_name = "AvitoTech_avibe.zip"

# Создаём ZIP-архив
shutil.make_archive(zip_name.replace('.zip', ''), 'zip', folder_path)

# Отображаем ссылку для скачивания
display(FileLink(zip_name))

/home/user-17/AvitoTech_avibe.zip

In [ ]:
padding_side = "right"

In [1]:
"""
Обучение LLM с помощью QLoRA для классификации запросов.
Исправленная версия: стабильный 8-bit LoRA + корректный Trainer + O(N) evaluation.
Добавлены детальный промпт и графики (сравнение accuracy, confusion matrices).
"""

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import gc
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    BitsAndBytesConfig,
    EarlyStoppingCallback,
    AutoModelForSequenceClassification
)

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

# ================== НАСТРОЙКИ ==================
TRAIN_PATH = "train.csv"
TEST_PATH = "test.csv"

MAX_LENGTH = 620
OUTPUT_DIR = "./llm_classifier"
PLOTS_DIR = "./llm_plots"
RESULTS_CSV = "results_right.csv"

os.makedirs(PLOTS_DIR, exist_ok=True)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

MODEL_NAMES = [
    'mistralai/Mistral-7B-Instruct-v0.3',
    "apple/SimpleSD-4B-instruct",
    "Qwen/Qwen3-4B-Instruct-2507",
    "Qwen/Qwen2.5-7B-Instruct",
    "unsloth/Meta-Llama-3.1-8B-Instruct",
    "openchat/openchat-3.6-8b-20240522",
    "google/gemma-4-E2B-it",
    "google/gemma-4-E4B-it",
    "AvitoTech/avibe",
    "OpenPipe/Qwen3-14B-Instruct",
    "unsloth/Mistral-Nemo-Instruct-2407",
    'ai-sage/GigaChat3.1-10B-A1.8B-bf16',
    'yandex/YandexGPT-5-Lite-8B-instruct'
]

DIGIT2CLASS = {
    "1": "теоретический вопрос",
    "2": "анализ данных",
    "3": "взаимодействие с графом",
    "4": "анализ данных и взаимодействие с графом",
    "5": "чушь"
}

CLASS_NAMES = list(DIGIT2CLASS.values())
# ================== DATA ==================
print("=" * 60)
print("ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ")
print("=" * 60)

df_train = pd.read_csv(TRAIN_PATH, sep=';', encoding='windows-1251').dropna()
df_test_raw = pd.read_csv(TEST_PATH, sep=';', encoding='windows-1251').dropna()

# ================== LABEL MAPPING ==================
def label_to_digit(label_str):
    s = str(label_str).strip()

    # уже цифра
    if s in DIGIT2CLASS:
        return s

    # текст → цифра
    inv_map = {v: k for k, v in DIGIT2CLASS.items()}
    return inv_map.get(s, None)

df_train["digit"] = df_train["label"].apply(label_to_digit)
df_test_raw["digit"] = df_test_raw["label"].apply(label_to_digit)

# ================== FILTER ==================
df_train = df_train[df_train["digit"].notna()]
df_test_raw = df_test_raw[df_test_raw["digit"].notna()]

# защита от пустого датасета
assert len(df_test_raw) > 0, "df_test_raw is EMPTY after label mapping"

# ================== SPLIT ==================
test_val, test_final = train_test_split(
    df_test_raw,
    test_size=0.65,
    random_state=42,
    stratify=df_test_raw["digit"]
)

print(f"Train size: {len(df_train)}")
print(f"Validation size: {len(test_val)}")
print(f"Final test size: {len(test_final)}")

# ================== DATASET ==================
def df_to_dataset(df):
    return Dataset.from_dict({
        "query": df["query"].tolist(),
        "digit": df["digit"].tolist()
    })

dataset_dict = DatasetDict({
    "train": df_to_dataset(df_train),
    "validation": df_to_dataset(test_val),
    "test": df_to_dataset(test_final)
})

val_queries = dataset_dict["validation"]["query"]
val_digits = dataset_dict["validation"]["digit"]
# ================== TOKENIZER ==================
def tokenize(examples, tokenizer):
    enc = tokenizer(
        examples["query"],
        truncation=True,
        #padding="max_length",
        max_length=MAX_LENGTH,
    )
    enc["labels"] = [int(d) - 1 for d in examples["digit"]]  # 0..4
    return enc
# ================== METRICS ==================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "f1": f1}


print("Train distribution:")
print(df_train["digit"].value_counts(normalize=True))

print("Validation distribution:")
print(test_val["digit"].value_counts(normalize=True))
# ================== TRAIN ==================
def train_llm(model_name):

    print(f"\n=== {model_name} ===")

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    #8-bit QLoRA config
    # bnb_config = BitsAndBytesConfig(
    #     load_in_8bit=True
    # )

    model_kwargs = {
        "device_map": "auto",
        "num_labels": 5,
        "torch_dtype": torch.bfloat16,
        "trust_remote_code": True,
        "low_cpu_mem_usage": True,
    }

    if "GigaChat" not in model_name:
        model_kwargs["attn_implementation"] = "sdpa" # <-- Магия происходит здесь
    else:
        print(f"⚠️ Для {model_name} оптимизация отключена во избежание конфликтов.")    

    
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        #device_map='auto',
        #num_labels=5,
        #quantization_config=bnb_config,
        #torch_dtype=torch.bfloat16,
        #trust_remote_code=True,
        #low_cpu_mem_usage=True,
        **model_kwargs
    )
    
    
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False
    #model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.035,
        bias="none",
        task_type="SEQ_CLS",
        target_modules="all-linear")
    
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # tokenize
    tokenized = dataset_dict.map(
        lambda x: tokenize(x, tokenizer),
        batched=True,
        remove_columns=dataset_dict["train"].column_names
    )

    data_collator = DataCollatorWithPadding(tokenizer)

    args = TrainingArguments(
        output_dir=os.path.join(OUTPUT_DIR, model_name.replace("/", "_")),
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=2,
        num_train_epochs=5,
        #fp16=True,
        bf16=True,
        optim="adamw_8bit",
        logging_steps=10,
        save_total_limit=3,
        metric_for_best_model="f1",
        load_best_model_at_end=True,
        greater_is_better=True,
        report_to="none",
        warmup_ratio=0.12,
        remove_unused_columns=False,
        seed=SEED,
    )


    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(2)],)

    print("TRAIN START")
    trainer.train()

    model.save_pretrained(args.output_dir)
    tokenizer.save_pretrained(args.output_dir)

    # Оценка на тестовом наборе
    result = trainer.evaluate(tokenized["test"])
    print("Test result:", result)
    
    # Предсказания
    pred_output = trainer.predict(tokenized["test"])
    
    logits = pred_output.predictions      # (N, 5)
    true_labels = pred_output.label_ids   # (N,)
    pred_labels = np.argmax(logits, axis=1)
    
    # очистка памяти
    trainer.model = None
    del trainer
    del model
    torch.cuda.synchronize()
    gc.collect()
    torch.cuda.empty_cache()
    
    return {
        "result": result,
        "true_labels": true_labels,
        "pred_labels": pred_labels,
        "model_name": model_name,
    }    
# ================== RUN ==================
all_info = []   # будет хранить словари с result, pred_labels, true_labels, model_name

for m in MODEL_NAMES:
    try:
        info = train_llm(m)
        all_info.append(info)
    except Exception as e:
        print(f"FAILED {m}: {e}")

# Сохраняем только числовые метрики в CSV
results_rows = []
for info in all_info:
    res = info["result"]
    res["model"] = info["model_name"]
    results_rows.append(res)
pd.DataFrame(results_rows).to_csv(RESULTS_CSV, index=False)
print("Results saved to", RESULTS_CSV)

# ================== ГРАФИКИ ==================
if all_info:
    # 1. График сравнения точности моделей
    plt.figure(figsize=(14, 8))

    # Подготовка данных
    models = [info["model_name"] for info in all_info]
    accs = [info["result"]["eval_accuracy"] for info in all_info]
    f1s = [info["result"]["eval_f1"] for info in all_info]

    comparison_df = pd.DataFrame({
        'Model': models,
        'Accuracy': accs,
        'F1-Score': f1s
    })

    # Сортируем модели по точности
    sorted_models = comparison_df.sort_values('Accuracy', ascending=True)

    # Создаем столбчатую диаграмму с двумя метриками
    x = range(len(sorted_models))
    width = 0.35

    plt.barh([i - width/2 for i in x], sorted_models['Accuracy'],
            width, label='Accuracy', color='skyblue', edgecolor='navy')
    plt.barh([i + width/2 for i in x], sorted_models['F1-Score'],
            width, label='F1-Score', color='lightcoral', edgecolor='darkred')

    # Добавляем значения на столбцы
    for i, (acc, f1) in enumerate(zip(sorted_models['Accuracy'], sorted_models['F1-Score'])):
        plt.text(acc + 0.01, i - width/2, f'{acc:.3f}',
                va='center', fontsize=9, fontweight='bold', color='navy')
        plt.text(f1 + 0.01, i + width/2, f'{f1:.3f}',
                va='center', fontsize=9, fontweight='bold', color='darkred')

    plt.yticks(x, sorted_models['Model'])
    plt.xlabel('Значение метрики', fontsize=12)
    plt.title('Сравнение производительности моделей',
             fontsize=16, fontweight='bold', pad=20)
    plt.legend(loc='lower right')
    plt.grid(axis='x', alpha=0.3)
    plt.xlim([0, 1.1])

    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, "models_comparison_right.png"), dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ График сравнения сохранен: {PLOTS_DIR}/models_comparison.png")

    # 2. Confusion matrices для каждой модели
    for info in all_info:
        model_name = info["model_name"]
        true = info["true_labels"]
        pred = info["pred_labels"]
        cm = confusion_matrix(true, pred, labels=list(range(len(CLASS_NAMES))))
        cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)
        cm_norm = np.nan_to_num(cm_norm)

        # Абсолютная матрица
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
        plt.title(f"Confusion Matrix: {model_name}")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, f"cm_{model_name.replace('/','_')}_right.png"), dpi=300)
        plt.close()

        # Нормализованная матрица
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm_norm, annot=True, fmt=".2%", cmap="YlOrRd",
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
        plt.title(f"Normalized CM: {model_name}")
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, f"cm_norm_{model_name.replace('/','_')}_right.png"), dpi=300)
        plt.close()

    print(f"All plots saved to {PLOTS_DIR}")

print("DONE")

ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ
Train size: 1608
Validation size: 237
Final test size: 442
Train distribution:
digit
3    0.226368
1    0.199005
5    0.193408
2    0.190920
4    0.190299
Name: proportion, dtype: float64
Validation distribution:
digit
5    0.261603
1    0.240506
3    0.172996
4    0.168776
2    0.156118
Name: proportion, dtype: float64

=== mistralai/Mistral-7B-Instruct-v0.3 ===


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] MistralForSequenceClassification LOAD REPORT from: mistralai/Mistral-7B-Instruct-v0.3
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 20,992,000 || all params: 7,134,818,304 || trainable%: 0.2942


...

Map:   0%|          | 0/442 [00:00<?, ? examples/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


TRAIN START


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.956812,0.472288,0.886076,0.881865
2,0.655880,0.216687,0.932489,0.928761


KeyboardInterrupt: 

In [5]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import gc
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    BitsAndBytesConfig,
    EarlyStoppingCallback,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

# ================== НАСТРОЙКИ ==================
TRAIN_PATH = "train.csv"
TEST_PATH = "test.csv"

MAX_LENGTH = 512
OUTPUT_DIR = "./llm_classifier"
PLOTS_DIR = "./llm_plots"
RESULTS_CSV = "gemma_finetune_results.csv"

os.makedirs(PLOTS_DIR, exist_ok=True)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

MODEL_NAMES = [
    "google/gemma-4-E2B-it",
    "google/gemma-4-E4B-it",
]

DIGIT2CLASS = {
    "1": "теоретический вопрос",
    "2": "анализ данных",
    "3": "взаимодействие с графом",
    "4": "анализ данных и взаимодействие с графом",
    "5": "чушь"
}
CLASS_NAMES = list(DIGIT2CLASS.values())

# Единый шаблон промпта для обучения и инференса
PROMPT_TEMPLATE = """Ты — классификатор запросов для no-code ML платформы. Верни только одну цифру.

1 — теория ML/математика/статистика.
2 — анализ данных.
3 — действия с графом ML-пайплайна.
4 — и анализ данных, и граф.
5 — нерелевантно.

Примеры:
"что такое регуляризация в линейной регрессии?" -> 1
"посмотри корреляцию признаков и найди выбросы" -> 2
"добавь датасет, label encoder и linear regression и соедини их" -> 3
"проанализируй датасет и добавь блок нормализации в граф" -> 4
"сколько времени в Токио?" -> 5
"привет, как дела?" -> 5

Верни только цифру.
Запрос: {query}
Ответ:"""

# ================== ДАННЫЕ ==================
print("=" * 60)
print("ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ")
print("=" * 60)

df_train = pd.read_csv(TRAIN_PATH, sep=';', encoding='windows-1251').dropna()
df_test_raw = pd.read_csv(TEST_PATH, sep=';', encoding='windows-1251').dropna()

def label_to_digit(label_str):
    s = str(label_str).strip()
    if s in DIGIT2CLASS:
        return s
    inv_map = {v: k for k, v in DIGIT2CLASS.items()}
    return inv_map.get(s, None)

df_train["digit"] = df_train["label"].apply(label_to_digit)
df_test_raw["digit"] = df_test_raw["label"].apply(label_to_digit)

df_train = df_train[df_train["digit"].notna()]
df_test_raw = df_test_raw[df_test_raw["digit"].notna()]

assert len(df_test_raw) > 0, "df_test_raw ПУСТ после маппинга лейблов!"

test_val, test_final = train_test_split(
    df_test_raw,
    test_size=0.65,
    random_state=SEED,
    stratify=df_test_raw["digit"]
)

def df_to_dataset(df):
    return Dataset.from_dict({
        "query": df["query"].tolist(),
        "digit": df["digit"].tolist()
    })

dataset_dict = DatasetDict({
    "train": df_to_dataset(df_train),
    "validation": df_to_dataset(test_val),
    "test": df_to_dataset(test_final)
})

# ================== ТОКЕНИЗАЦИЯ С МАСКИРОВАНИЕМ ==================
def tokenize_function(examples, tokenizer):
    input_ids_list = []
    attention_mask_list = []
    labels_list = []

    for q, d in zip(examples["query"], examples["digit"]):
        prompt_text = PROMPT_TEMPLATE.format(query=q)
        full_text = prompt_text + f" {d}"

        enc_prompt = tokenizer(prompt_text, truncation=True, max_length=MAX_LENGTH)
        enc_full = tokenizer(full_text, truncation=True, max_length=MAX_LENGTH)

        prompt_len = len(enc_prompt["input_ids"])
        
        # Маскируем токенами -100 всё, кроме целевой цифры ответа
        labels = [-100] * prompt_len + enc_full["input_ids"][prompt_len:]

        input_ids_list.append(enc_full["input_ids"])
        attention_mask_list.append(enc_full["attention_mask"])
        labels_list.append(labels)

    # Ручной Left-Padding для стабильной работы CausalLM при обучении и генерации
    padded_input_ids = []
    padded_attention = []
    padded_labels = []

    for ids, am, lb in zip(input_ids_list, attention_mask_list, labels_list):
        pad_len = MAX_LENGTH - len(ids)
        if pad_len > 0:
            padded_input_ids.append([tokenizer.pad_token_id] * pad_len + ids)
            padded_attention.append([0] * pad_len + am)
            padded_labels.append([-100] * pad_len + lb)
        else:
            padded_input_ids.append(ids[:MAX_LENGTH])
            padded_attention.append(am[:MAX_LENGTH])
            padded_labels.append(lb[:MAX_LENGTH])

    return {
        "input_ids": padded_input_ids,
        "attention_mask": padded_attention,
        "labels": padded_labels
    }

# ================== ОПТИМИЗИРОВАННЫЙ ИНФЕРЕНС (O(N) BATCHED) ==================
def evaluate_generation(model, tokenizer, dataset, batch_size=8):
    model.eval()
    true_labels = []
    pred_labels = []

    queries = dataset["query"]
    digits = dataset["digit"]

    for i in range(0, len(queries), batch_size):
        batch_queries = queries[i : i + batch_size]
        batch_digits = digits[i : i + batch_size]

        prompts = [PROMPT_TEMPLATE.format(query=q) for q in batch_queries]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=2, 
                do_sample=False,
                temperature=0.0,
            )

        input_len = inputs["input_ids"].shape[1]
        for j, out in enumerate(outputs):
            gen_tokens = out[input_len:]
            decoded = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

            # Ищем первую попавшуюся валидную цифру в сгенерированном ответе
            pred_digit = -1 
            for c in decoded:
                if c in "12345":
                    pred_digit = int(c) - 1
                    break

            # Если модель выдала бред, отправляем в 5-й класс ("чушь", индекс 4)
            if pred_digit == -1:
                pred_digit = 4 

            true_labels.append(int(batch_digits[j]) - 1)
            pred_labels.append(pred_digit)

    acc = accuracy_score(true_labels, pred_labels)
    f1 = f1_score(true_labels, pred_labels, average="macro")

    return {
        "accuracy": acc,
        "f1": f1,
        "true_labels": np.array(true_labels),
        "pred_labels": np.array(pred_labels),
    }

# ================== ОБУЧЕНИЕ МОДЕЛИ ==================
def train_llm(model_name):
    print(f"\n=== ЗАПУСК: {model_name} ===")

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        attn_implementation="sdpa"
    )
    model.config.use_cache = False

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.035,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules="all-linear",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # Токенизируем данные (исходные текстовые колонки остаются нетронутыми в dataset_dict)
    tokenized_train = dataset_dict["train"].map(lambda x: tokenize_function(x, tokenizer), batched=True)
    tokenized_val = dataset_dict["validation"].map(lambda x: tokenize_function(x, tokenizer), batched=True)

    data_collator = DataCollatorForSeq2Seq(
        tokenizer, 
        model=model, 
        label_pad_token_id=-100, 
        padding="longest"
    )
    
    args = TrainingArguments(
        output_dir=os.path.join(OUTPUT_DIR, model_name.replace("/", "_")),
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=2,
        num_train_epochs=5,
        bf16=True,
        optim="adamw_8bit",
        logging_steps=10,
        save_total_limit=3,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        load_best_model_at_end=True,
        report_to="none",
        warmup_ratio=0.1,
        seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        callbacks=[EarlyStoppingCallback(2)],
        data_collator=data_collator
    )
    
    print("СТАРТ ОБУЧЕНИЯ")
    trainer.train()
    
    print("СОХРАНЕНИЕ ОБУЧЕННОЙ LORA-МОДЕЛИ И ТОКЕНИЗАТОРА")
    trainer.save_model(args.output_dir) 
    tokenizer.save_pretrained(args.output_dir)

    print("СТАРТ ТЕСТИРОВАНИЯ")

    # Оценка на тестовом наборе (батчами)
    print("СТАРТ ГЕНЕРАЦИОННОГО ТЕСТИРОВАНИЯ")
    test_eval = evaluate_generation(
        model=model,
        tokenizer=tokenizer,
        dataset=dataset_dict["test"],
        batch_size=8
    )
    
    result = {
        "eval_accuracy": test_eval["accuracy"],
        "eval_f1": test_eval["f1"],
    }
    print(f"Результаты для {model_name}: {result}")
        
    # Чистим VRAM
    del trainer
    del model
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

    return {
        "result": result,
        "true_labels": test_eval["true_labels"],
        "pred_labels": test_eval["pred_labels"],
        "model_name": model_name
    }

# ================== RUN И ОТРИСОВКА ГРАФИКОВ ==================
all_info = []

for m in MODEL_NAMES:
    try:
        info = train_llm(m)
        all_info.append(info)
    except Exception as e:
        print(f"Ошибка при работе с моделью {m}: {e}")

if all_info:
    results_rows = []
    for info in all_info:
        res = info["result"].copy()
        res["model"] = info["model_name"]
        results_rows.append(res)
    pd.DataFrame(results_rows).to_csv(RESULTS_CSV, index=False)
    print("Метрики успешно сохранены в файл:", RESULTS_CSV)

    # 1. Сравнение моделей
    plt.figure(figsize=(14, 8))
    models = [info["model_name"] for info in all_info]
    accs = [info["result"]["eval_accuracy"] for info in all_info]
    f1s = [info["result"]["eval_f1"] for info in all_info]

    comparison_df = pd.DataFrame({'Model': models, 'Accuracy': accs, 'F1-Score': f1s}).sort_values('Accuracy', ascending=True)

    x = range(len(comparison_df))
    width = 0.35

    plt.barh([i - width/2 for i in x], comparison_df['Accuracy'], width, label='Accuracy', color='skyblue', edgecolor='navy')
    plt.barh([i + width/2 for i in x], comparison_df['F1-Score'], width, label='F1-Score', color='lightcoral', edgecolor='darkred')

    for i, (acc, f1) in enumerate(zip(comparison_df['Accuracy'], comparison_df['F1-Score'])):
        plt.text(acc + 0.01, i - width/2, f'{acc:.3f}', va='center', fontsize=9, fontweight='bold', color='navy')
        plt.text(f1 + 0.01, i + width/2, f'{f1:.3f}', va='center', fontsize=9, fontweight='bold', color='darkred')

    plt.yticks(x, comparison_df['Model'])
    plt.xlabel('Значение метрики', fontsize=12)
    plt.title('Сравнение производительности моделей', fontsize=16, fontweight='bold', pad=20)
    plt.legend(loc='lower right')
    plt.grid(axis='x', alpha=0.3)
    plt.xlim([0, 1.1])
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, "models_comparison_gemma.png"), dpi=300, bbox_inches='tight')
    plt.close()

    # 2. Confusion Matrices
    for info in all_info:
        model_name = info["model_name"]
        true = info["true_labels"]
        pred = info["pred_labels"]
        
        cm = confusion_matrix(true, pred, labels=list(range(len(CLASS_NAMES))))
        with np.errstate(all='ignore'):
            cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)
            cm_norm = np.nan_to_num(cm_norm)

        # Абсолютная
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
        plt.title(f"Confusion Matrix: {model_name}")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, f"cm_{model_name.replace('/','_')}.png"), dpi=300)
        plt.close()

        # Нормализованная
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm_norm, annot=True, fmt=".2%", cmap="YlOrRd", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
        plt.title(f"Normalized CM: {model_name}")
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, f"cm_norm_{model_name.replace('/','_')}.png"), dpi=300)
        plt.close()

    print(f"Все графики успешно сохранены в папку: {PLOTS_DIR}")

print("ГОТОВО!")

ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ

=== ЗАПУСК: google/gemma-4-E2B-it ===


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

trainable params: 37,920,768 || all params: 5,142,218,272 || trainable%: 0.7374


Map:   0%|          | 0/1608 [00:00<?, ? examples/s]

Map:   0%|          | 0/237 [00:00<?, ? examples/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


СТАРТ ОБУЧЕНИЯ
Ошибка при работе с моделью google/gemma-4-E2B-it: CUDA out of memory. Tried to allocate 48.00 MiB. GPU 0 has a total capacity of 47.37 GiB of which 23.00 MiB is free. Process 279469 has 15.56 GiB memory in use. Including non-PyTorch memory, this process has 31.36 GiB memory in use. Of the allocated memory 30.84 GiB is allocated by PyTorch, and 22.17 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

=== ЗАПУСК: google/gemma-4-E4B-it ===


[transformers] Current model requires 6192 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

KeyboardInterrupt: 